# LeetCode 21 — Merge Two Sorted Lists | Nhập môn Python, giảng từng bước

**Độ khó:** Easy · **Kỹ thuật trọng tâm:** Linked List · con trỏ · dummy node  
**Bài gốc:** https://leetcode.com/problems/merge-two-sorted-lists/

Notebook này dành cho người **mới bắt đầu**. Đọc các ô từ trên xuống: trước hết hiểu **vì sao**, sau đó chạy code nhỏ để tự thấy **cách làm**.

**Khung hàm (signature) cần nộp LeetCode:**

```python
# LeetCode cung cấp lớp ListNode
class Solution:
    def mergeTwoLists(self, list1: Optional[ListNode], list2: Optional[ListNode]) -> Optional[ListNode]:
        ...
```

**Sau bài học, bạn có thể:**
- Tự tạo, duyệt và vẽ được một danh sách liên kết (linked list).
- Hiểu tại sao phải di chuyển con trỏ thay vì gộp rồi sort.
- Viết được thuật toán gộp bằng dummy node với O(m+n) thời gian, O(1) nhớ phụ.

**Cách sử dụng:** chạy từng code cell theo thứ tự bằng `Shift + Enter`. Những ô có `TODO` là bài tự luyện: chỉ định nghĩa hàm hoặc chứa lời nhắc kiểm thử, **không được gọi tự động** khi chưa làm xong.

> Lộ trình: hiểu đề → Python căn bản → cách trực giác → phát hiện nút thắt → cải tiến → trace → nộp LeetCode → test → lỗi → vận dụng → bài tập → đáp án → checklist.

## 1. Đề bài — gộp hai danh sách đã sắp xếp

Cho **đầu (head)** của hai linked list `list1`, `list2`, mỗi danh sách đã được sắp xếp **không giảm** (cho phép số trùng). Hãy **nối lại các node có sẵn** thành một linked list duy nhất cũng không giảm, và **trả head mới**.

Ví dụ:

```text
list1: 1 → 2 → 4 → None
list2: 1 → 3 → 4 → None
kết quả: 1 → 1 → 2 → 3 → 4 → 4 → None
```

Nếu một danh sách rỗng (`None`), ta trả danh sách còn lại. **Không** trả list Python `[1,1,2,...]`, vì LeetCode yêu cầu `ListNode`.

**Phân biệt:** linked list có các node nối bằng `.next`, khác với `list` Python chứa phần tử truy cập bằng chỉ số.

## 2. `ListNode` là gì?

Mỗi **node** có hai ô: `val` (giá trị) và `next` (tham chiếu đến node kế tiếp). `None` nghĩa là **không có node tiếp theo**. `head` là biến trỏ đến node đầu tiên.

Ví dụ: `a = ListNode(1); b = ListNode(3); a.next = b` tạo đường nối `1 → 3 → None`.

Trong LeetCode, lớp `ListNode` **được cho sẵn**. Trong Jupyter ta phải định nghĩa một phiên bản tương đương để code chạy được độc lập.

In [ ]:
from __future__ import annotations
from typing import Optional

class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

first = ListNode(1)
second = ListNode(3)
first.next = second
print("Node đầu:", first.val, "| node kế:", first.next.val)
print("Cuối list:", second.next)

## 3. Thực hành `head` và `.next`

Đi theo mũi tên `.next` từ `head` cho đến `None`. Đây là phép **traversal (duyệt)** một linked list. Không có chỉ số `head[0]` như list Python.

Mỗi lần `current = current.next`, biến `current` chỉ **chuyển sang node tiếp theo**; nó không xóa node cũ.

In [ ]:
head = ListNode(2, ListNode(5, ListNode(9)))
current = head
while current is not None:
    print("Đang ở node:", current.val)
    current = current.next
print("Duyệt xong:", current)

## 4. Hai hàm phụ giúp nhìn dữ liệu

Ta cần hàm `build_list([1,2,4])` biến list Python thành linked list để chạy ví dụ và `to_values(head)` chuyển ngược về list Python **để hiển thị/test**. Hai hàm này chỉ phục vụ học tập, **không** cần dán lên LeetCode.

In [ ]:
def build_list(values: list[int]) -> Optional[ListNode]:
    head = None
    for value in reversed(values):
        head = ListNode(value, head)
    return head

def to_values(head: Optional[ListNode]) -> list[int]:
    values = []
    current = head
    while current is not None:
        values.append(current.val)
        current = current.next
    return values

head = build_list([1, 2, 4])
print("Vẽ theo chiều mũi tên:", " -> ".join(map(str, to_values(head))), "-> None")

## 5. Python căn bản — `None`, `is`, `while`

- `None` đại diện cho **không có object/node**. `node is None` kiểm tra biến có đang chỉ vào “không có node” hay không.
- `while condition:` lặp khi điều kiện còn đúng.
- `while a is not None and b is not None` chỉ chạy khi **cả hai** danh sách còn node.
- `a.val` đọc giá trị; `a.next` đọc **đường nối**.

**Lưu ý:** `a = a.next` chỉ di chuyển biến `a`; còn `tail.next = a` mới **sửa mũi tên** trong linked list.

In [ ]:
node = ListNode(7)
print("node có tồn tại?", node is not None)
print("node.next chưa có?", node.next is None)
node.next = ListNode(10)
print("Sau khi nối:", to_values(node))

## 6. Trực giác: chỉ so hai node đầu

Hai danh sách **đã có thứ tự** nên phần tử nhỏ nhất chưa dùng phải nằm ở **một trong hai đầu danh sách hiện tại**.

Ví dụ `A = 1→2→4`, `B = 1→3→4`: so `1` với `1` ⇒ lấy A trước; sau đó so `2` với `1` ⇒ lấy B; tiếp theo `2` với `3` ⇒ lấy A... Không cần sắp xếp lại tất cả.

Dùng cách lặp lại tương tự hai bạn đang lần lượt lấy thẻ nhỏ nhất từ hai chồng thẻ **đều đã xếp tăng**.

## 7. Cách đơn giản đầu tiên — đệ quy

**Recursion (đệ quy)** là hàm gọi lại chính nó với bài toán nhỏ hơn. Nếu `a` hoặc `b` rỗng, trả ngay danh sách còn lại. Nếu không, chọn node nhỏ hơn làm đầu và gộp hai phần còn lại làm `.next`.

Đệ quy ngắn nhưng sử dụng **call stack** (ngăn xếp lời gọi) có thể sâu `m+n`, nên tối đa **O(m+n)** bộ nhớ phụ và có nguy cơ vượt giới hạn đệ quy ở Python khi đầu vào dài. Ta sẽ cải tiến bằng vòng lặp.

In [ ]:
def merge_recursive(a: Optional[ListNode], b: Optional[ListNode]) -> Optional[ListNode]:
    if a is None:
        return b
    if b is None:
        return a
    if a.val <= b.val:
        a.next = merge_recursive(a.next, b)
        return a
    b.next = merge_recursive(a, b.next)
    return b

result = merge_recursive(build_list([1, 2, 4]), build_list([1, 3, 4]))
print("Đệ quy:", to_values(result))

## 8. Tại sao `dummy` (node giả) giúp code dễ hơn?

Để nối từng node vào **cuối danh sách kết quả**, ta cần biết **node cuối hiện tại** (`tail`). Nhưng lúc đầu chưa có node kết quả, dễ phải viết `if head is None` trong mỗi lần thêm.

Mẹo: tạo node giả `dummy = ListNode()` chỉ làm **mốc khởi đầu**. Cho `tail = dummy` rồi cứ nối `tail.next = chosen` và cập nhật `tail = tail.next`. Kết quả thật bắt đầu ở `dummy.next`.

```text
dummy(giả) → 1 → 2 → 3 → ...
                 ↑ tail đang ở node cuối mới nối
```

`dummy` **không** thuộc kết quả cuối; ta trả `dummy.next`, không trả `dummy`.

In [ ]:
dummy = ListNode(-1)
tail = dummy
node_a = ListNode(2)
tail.next = node_a
print("Sau nối A:", to_values(dummy.next))
tail = tail.next
node_b = ListNode(5)
tail.next = node_b
print("Sau nối B:", to_values(dummy.next))

## 9. Di chuyển `a`, `b` và `tail` thế nào?

Giả sử `a.val <= b.val`. Ta nối node `a` vào đuôi: `tail.next = a`; **sau đó di chuyển con trỏ nguồn** `a = a.next`, rồi **di chuyển đuôi** `tail = tail.next`.

Đường nối cũ của node vừa chọn sẽ bị thay ở lần ghép tiếp theo. Luôn chỉ lấy từ một bên mỗi vòng, không được quên tăng đúng con trỏ nguồn. Khi một bên hết, bên còn lại vốn đã tăng dần ⇒ có thể nối nguyên đoạn bằng `tail.next = a if a else b`.

In [ ]:
a = build_list([1, 4])
b = build_list([2, 3])
dummy = ListNode()
tail = dummy
print("Bắt đầu:", to_values(a), to_values(b))
tail.next = a
# Lưu con trỏ đến node tiếp theo trước khi thay đường nối về sau
a = a.next
tail = tail.next
print("Lấy 1 -> kết quả:", to_values(dummy.next), "| A còn:", to_values(a))

## 10. Bảng trace bằng tay

Cho A=`1→2→4`, B=`1→3→4`.

| Vòng | đầu A | đầu B | lấy | kết quả tạm |
|---:|---:|---:|---:|---|
| 1 | 1 | 1 | A:1 | 1 |
| 2 | 2 | 1 | B:1 | 1→1 |
| 3 | 2 | 3 | A:2 | 1→1→2 |
| 4 | 4 | 3 | B:3 | 1→1→2→3 |
| 5 | 4 | 4 | A:4 | 1→1→2→3→4 |
| hết A | - | 4 | nối nguyên B | 1→1→2→3→4→4 |

**Bất biến:** kết quả đã nối luôn có thứ tự không giảm và chứa đúng các node nhỏ nhất đã lấy ra từ 2 danh sách.

## 11. Code chính — vòng lặp, O(1) bộ nhớ phụ

Mỗi lần so `a.val <= b.val`, lấy **node hiện có**, không tạo `ListNode` mới cho giá trị đó. Chỉ `dummy` là node mới để làm mốc.

In [ ]:
def merge_iterative(a: Optional[ListNode], b: Optional[ListNode]) -> Optional[ListNode]:
    dummy = ListNode()
    tail = dummy
    while a is not None and b is not None:
        if a.val <= b.val:
            tail.next = a
            a = a.next
        else:
            tail.next = b
            b = b.next
        tail = tail.next
    tail.next = a if a is not None else b
    return dummy.next

print(to_values(merge_iterative(build_list([1, 2, 4]), build_list([1, 3, 4]))))
print(to_values(merge_iterative(build_list([]), build_list([0]))))

## 12. Trace code — theo dõi từng con trỏ

Bản dưới in **hai đầu còn lại**, node được lấy và **kết quả hiện tại**. Bảng trace với node chứa giá trị bằng nhau cho thấy quy tắc `<=` ưu tiên bên trái; nếu dùng `<` thì vẫn có thể đúng vì hai giá trị bằng nhau, nhưng thứ tự node nguồn có thể khác.

In [ ]:
def trace_merge(a, b):
    dummy = ListNode(); tail = dummy; step = 0
    while a is not None and b is not None:
        step += 1
        left = a.val; right = b.val
        if a.val <= b.val:
            taken = a.val; tail.next = a; a = a.next; side = "A"
        else:
            taken = b.val; tail.next = b; b = b.next; side = "B"
        tail = tail.next
        # Chỉ đọc phần kết quả đến tail, tránh đường nối cũ chưa ghi đè
        vals = []; curr = dummy.next
        while curr is not tail:
            vals.append(curr.val); curr = curr.next
        vals.append(tail.val)
        print(f"Bước {step}: A={left}, B={right} -> lấy {side}:{taken} | kết quả: {vals}")
    tail.next = a if a is not None else b
    print("Nối đoạn còn lại:", to_values(dummy.next))
    return dummy.next

assert to_values(trace_merge(build_list([1, 2, 4]), build_list([1, 3, 4]))) == [1, 1, 2, 3, 4, 4]

## 13. Code nộp LeetCode đúng signature

Trong ô nộp thật, **LeetCode đã định nghĩa `ListNode`**. Nếu chạy ở Jupyter, phiên bản `ListNode` ta đã tạo ở phần đầu sẽ được sử dụng. `Optional[ListNode]` nghĩa là có thể nhận node **hoặc** `None`.

**Không dán các hàm `build_list`, `to_values` hay trace** khi nộp.

In [ ]:
class Solution:
    def mergeTwoLists(self, list1: Optional[ListNode], list2: Optional[ListNode]) -> Optional[ListNode]:
        dummy = ListNode()
        tail = dummy
        while list1 is not None and list2 is not None:
            if list1.val <= list2.val:
                tail.next = list1
                list1 = list1.next
            else:
                tail.next = list2
                list2 = list2.next
            tail = tail.next
        tail.next = list1 if list1 is not None else list2
        return dummy.next

print("Submit demo:", to_values(Solution().mergeTwoLists(build_list([1, 2]), build_list([3]))))

## 14. Tại sao đúng? Thời gian và bộ nhớ

**Bất biến:** trước mỗi vòng, các node đã nối vào kết quả luôn được sắp xếp tăng dần và là những node nhỏ nhất đã xét. Vì hai danh sách còn lại được sắp xếp, node nhỏ nhất còn lại chắc chắn ở **đầu A hoặc đầu B**. Lấy đầu nhỏ hơn không thể làm sai thứ tự; cuối cùng nối đoạn còn lại vốn đã tăng dần.

Gọi `m`, `n` là số node của hai danh sách. Mỗi vòng lặp lấy đúng **một** node, tổng không quá `m+n` vòng ⇒ **O(m+n)** thời gian. Chỉ dùng `dummy`, `tail`, `a`, `b` ⇒ **O(1)** bộ nhớ phụ. Đệ quy cũng O(m+n) thời gian nhưng **O(m+n)** bộ nhớ stack trong trường hợp xấu nhất.

**Không nhầm:** O(1) bộ nhớ phụ **không** tính các node đầu vào, vì ta dùng lại chúng.

## 15. Kiểm thử với mọi dạng list

Ngoài kiểm tra **giá trị kết quả**, ta kiểm tra luôn **danh tính node** (object identity) để chắc chắn thuật toán tái sử dụng các node input, chứ không âm thầm tạo danh sách mới.

In [ ]:
tests_21 = [
    ([1, 2, 4], [1, 3, 4], [1, 1, 2, 3, 4, 4]),
    ([], [], []), ([], [0], [0]), ([0], [], [0]),
    ([1, 1, 1], [1, 1], [1, 1, 1, 1, 1]),
    ([-5, 2], [-7, 3], [-7, -5, 2, 3]),
    ([1, 2], [3, 4, 5], [1, 2, 3, 4, 5]),
]

def all_node_ids(head):
    ids = []; curr = head
    while curr is not None:
        assert id(curr) not in ids, "Phát hiện vòng lặp bất thường"
        ids.append(id(curr)); curr = curr.next
    return ids

for l1, l2, expected in tests_21:
    a = build_list(l1); b = build_list(l2)
    original_ids = set(all_node_ids(a) + all_node_ids(b))
    merged = Solution().mergeTwoLists(a, b)
    assert to_values(merged) == expected
    assert set(all_node_ids(merged)) == original_ids
    recursive = merge_recursive(build_list(l1), build_list(l2))
    assert to_values(recursive) == expected
print("PASS", len(tests_21), "ca giá trị + tái sử dụng node + đối chiếu recursive")

## 16. Lỗi người mới: mất node, trả dummy, quên di chuyển

- `return dummy`: sai vì trả **node giả** thay vì kết quả thực → dùng `dummy.next`.
- `tail.next = a` nhưng **quên** `a = a.next`: dễ lặp vô hạn hoặc tạo vòng nối không đúng.
- `tail = tail.next` quá sớm/trước khi nối: cập nhật sai nơi ghép.
- Chỉ nối trong vòng `while` nhưng quên `tail.next = ...` khi một bên hết: mất **đoạn cuối**.
- Dùng `append` trên `ListNode`: `append` thuộc **list Python**, không có trên node.

Hãy tự vẽ 3 biến `a`, `b`, `tail` trên giấy sau mỗi vòng để phát hiện những lỗi này.

## 17. Khi nào nên nghĩ tới Two Pointers + dummy?

Dấu hiệu: hai đầu dữ liệu đã sắp xếp; cần ghép **không cấp phát lại**; phải thao tác với `.next`; đầu kết quả có thể thay đổi. Mẫu `dummy + tail` cũng xuất hiện ở Partition List, Remove Nth Node, Swap Nodes in Pairs.

**Mở rộng:** nếu các danh sách **không sắp xếp**, quy tắc “lấy đầu nhỏ hơn” không còn đúng. Vì vậy đừng dùng lại công thức khi không có tiền đề sorted.

## 18. Bài tập tự luyện — có 5 mức và gợi ý

### Bài 1 — Tạo node và xem đường nối — Tự làm

Viết hàm tạo linked list gồm đúng 2 node `x → y → None`.

**Gợi ý cấp 1 (tư duy):** Cần một head và node tiếp theo.

<details><summary>Gợi ý cấp 2 — Cú pháp</summary>

`ListNode(x)` và thuộc tính `.next`.

</details>

<details><summary>Gợi ý cấp 3 — Gần code</summary>

`head = ListNode(x); head.next = ListNode(y); return head`.

</details>

**Khi làm xong**, bỏ dấu `#` ở lệnh `assert` gợi ý trong ô code dưới để tự kiểm tra. Đáp án đầy đủ nằm ở phần cuối notebook.

In [ ]:
def exercise_21_1(x: int, y: int) -> ListNode:
    # TODO: tạo hai ListNode và nối bằng .next
    pass

# assert to_values(exercise_21_1(2, 5)) == [2, 5]

### Bài 2 — Duyệt linked list — Tự làm

Tự viết hàm tính tổng các `.val` bằng vòng `while`.

**Gợi ý cấp 1 (tư duy):** Khởi tạo biến tổng 0.

<details><summary>Gợi ý cấp 2 — Cú pháp</summary>

`while head is not None`, `head = head.next`.

</details>

<details><summary>Gợi ý cấp 3 — Gần code</summary>

Mỗi vòng `total += head.val` và di chuyển đến node sau.

</details>

**Khi làm xong**, bỏ dấu `#` ở lệnh `assert` gợi ý trong ô code dưới để tự kiểm tra. Đáp án đầy đủ nằm ở phần cuối notebook.

In [ ]:
def exercise_21_2(head: Optional[ListNode]) -> int:
    # TODO: duyệt đến None, cộng val
    pass

# assert exercise_21_2(build_list([1, 2, 3])) == 6

### Bài 3 — Ghép bằng đệ quy — Tự làm

Viết lại `merge_recursive` tự tay, chú ý hai ca dừng.

**Gợi ý cấp 1 (tư duy):** Mỗi lời gọi phải chọn một node.

<details><summary>Gợi ý cấp 2 — Cú pháp</summary>

`a.next = exercise_21_3(a.next, b)`.

</details>

<details><summary>Gợi ý cấp 3 — Gần code</summary>

Ca còn lại tương tự nhưng nối `b.next` rồi `return b`.

</details>

**Khi làm xong**, bỏ dấu `#` ở lệnh `assert` gợi ý trong ô code dưới để tự kiểm tra. Đáp án đầy đủ nằm ở phần cuối notebook.

In [ ]:
def exercise_21_3(a: Optional[ListNode], b: Optional[ListNode]) -> Optional[ListNode]:
    # TODO: if a is None: ...
    # TODO: if b is None: ...
    # TODO: so sánh val và gọi lại chính hàm
    pass

# assert to_values(exercise_21_3(build_list([1, 3]), build_list([2, 4]))) == [1, 2, 3, 4]

### Bài 4 — Ghép bằng dummy node — Tự làm

Tự viết phiên bản O(1) bộ nhớ với dummy, tail, hai con trỏ.

**Gợi ý cấp 1 (tư duy):** Sau mỗi vòng, tail là node cuối đã ghép.

<details><summary>Gợi ý cấp 2 — Cú pháp</summary>

`while a is not None and b is not None`.

</details>

<details><summary>Gợi ý cấp 3 — Gần code</summary>

Nhánh a nhỏ hơn: `tail.next = a; a = a.next`, sau đó `tail = tail.next`.

</details>

**Khi làm xong**, bỏ dấu `#` ở lệnh `assert` gợi ý trong ô code dưới để tự kiểm tra. Đáp án đầy đủ nằm ở phần cuối notebook.

In [ ]:
def exercise_21_4(a: Optional[ListNode], b: Optional[ListNode]) -> Optional[ListNode]:
    # TODO: tạo dummy + tail
    # TODO: while a và b đều có node
    # TODO: nối node nhỏ hơn, đi con trỏ nguồn và tail
    # TODO: nối phần còn lại, trả dummy.next
    pass

# assert to_values(exercise_21_4(build_list([1, 2, 4]), build_list([1, 3, 4]))) == [1, 1, 2, 3, 4, 4]

### Bài 5 — Mini challenge đếm node — Tự làm

Gộp 2 linked list sorted, rồi trả số node của kết quả. Có thể dùng lại hàm merge đã học (biến thể luyện duyệt node).

**Gợi ý cấp 1 (tư duy):** Mỗi lần đi qua một node, tăng counter.

<details><summary>Gợi ý cấp 2 — Cú pháp</summary>

Có thể dùng `merge_iterative(a, b)` trước.

</details>

<details><summary>Gợi ý cấp 3 — Gần code</summary>

`curr = merged; count = 0; while curr is not None: ...`.

</details>

**Khi làm xong**, bỏ dấu `#` ở lệnh `assert` gợi ý trong ô code dưới để tự kiểm tra. Đáp án đầy đủ nằm ở phần cuối notebook.

In [ ]:
def exercise_21_5(a: Optional[ListNode], b: Optional[ListNode]) -> int:
    # TODO: gộp + duyệt đếm node; không dùng len(to_values(...))
    pass

# assert exercise_21_5(build_list([1, 2]), build_list([1, 3, 4])) == 5

## 19. Đáp án tham khảo (đặt tách xa TODO)

In [ ]:
def answer_21_1(x, y):
    head = ListNode(x)
    head.next = ListNode(y)
    return head

def answer_21_2(head):
    total = 0
    while head is not None:
        total += head.val
        head = head.next
    return total

def answer_21_3(a, b):
    if a is None:
        return b
    if b is None:
        return a
    if a.val <= b.val:
        a.next = answer_21_3(a.next, b)
        return a
    b.next = answer_21_3(a, b.next)
    return b

def answer_21_4(a, b):
    dummy = ListNode(); tail = dummy
    while a is not None and b is not None:
        if a.val <= b.val:
            tail.next = a; a = a.next
        else:
            tail.next = b; b = b.next
        tail = tail.next
    tail.next = a if a is not None else b
    return dummy.next

def answer_21_5(a, b):
    merged = answer_21_4(a, b)
    count = 0
    while merged is not None:
        count += 1
        merged = merged.next
    return count

assert to_values(answer_21_1(2, 5)) == [2, 5]
assert answer_21_2(build_list([1, 2, 3])) == 6
assert to_values(answer_21_3(build_list([1, 3]), build_list([2, 4]))) == [1, 2, 3, 4]
assert to_values(answer_21_4(build_list([1, 2, 4]), build_list([1, 3, 4]))) == [1, 1, 2, 3, 4, 4]
assert answer_21_5(build_list([1, 2]), build_list([1, 3, 4])) == 5
print("Đáp án tham khảo: PASS 5/5")

## Tóm tắt — mẫu tư duy cần nhớ

**Mẫu:** hai đầu sorted → so giá trị đầu → nối node nhỏ hơn vào `tail.next` → dịch con trỏ nguồn → dịch tail → nối phần còn lại → trả `dummy.next`. **O(m+n)** thời gian, **O(1)** bộ nhớ phụ.

## Checklist trước khi submit

- [ ] Tôi phân biệt được list Python và ListNode.
- [ ] Tôi hiểu .next thay đổi liên kết còn `a=a.next` chỉ di chuyển con trỏ.
- [ ] Tôi biết dummy là node giả và phải trả dummy.next.
- [ ] Tôi xử lý được hai list rỗng và số trùng nhau.
- [ ] Tôi đã test output và tính tái sử dụng node.

**Tự kiểm tra lần cuối:** có thể giải thích lời giải bằng lời trước khi xem code? Có tự tính được ít nhất một bảng trace? Nếu chưa, quay lại phần mô phỏng.

*Notebook học tập được biên soạn theo cẩm nang tiếng Việt đính kèm; tiêu đề/đề/ràng buộc bám sát LeetCode. Những biến thể được đánh dấu là mở rộng, không phải yêu cầu đề chính.*